# COLAB - Chay Pipeline Big Data

Notebook nay danh rieng cho Colab de bao cao.

Quy trinh:
1. Clone repo
2. Cai thu vien
3. Nap kaggle.json
4. Chay pipeline
5. Xem ket qua RFM + MBA

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/phoudsavanhKongmany/final-bigdata-project-nhom9Test.git'
REPO_DIR = Path('/content/final-bigdata-project-nhom9Test')

if not REPO_DIR.exists():
    !git clone {REPO_URL}
else:
    print('Repo da ton tai, tien hanh cap nhat...')
    !git -C {REPO_DIR} pull

%cd /content/final-bigdata-project-nhom9Test
!pwd

In [ ]:
# Cai dependencies
!pip -q install -r requirements.txt

## Nap kaggle.json
Ban tai file `kaggle.json` tu tai khoan Kaggle, sau do upload bang cell duoi.

In [ ]:
from google.colab import files

uploaded = files.upload()

if 'kaggle.json' not in uploaded:
    raise RuntimeError('Ban can upload file kaggle.json de tiep tuc.')

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Da cau hinh Kaggle credentials.')

## Chay pipeline
Mac dinh ben duoi se chay ca MBA + RFM.

Luu y: `--mba-sample-fraction 0.2` de nhe RAM tren Colab free.

In [ ]:
!python3 src/main_pipeline.py --project both --step all --mba-sample-fraction 0.2 --min-support 0.01 --min-confidence 0.2

In [ ]:
# Kiem tra nhanh output da sinh ra chua
!ls -lah data/3_curated/results/rfm || true
!ls -lah data/3_curated/results/mba || true

## Xem ket qua RFM

In [ ]:
import pandas as pd

rfm_segments_path = Path('data/3_curated/results/rfm/customer_segments')
rfm_summary_path = Path('data/3_curated/results/rfm/segment_summary')

if rfm_segments_path.exists() and rfm_summary_path.exists():
    rfm_segments = pd.read_parquet(rfm_segments_path)
    rfm_summary = pd.read_parquet(rfm_summary_path)
    display(rfm_segments.head())
    display(rfm_summary.sort_values('business_score', ascending=False))
else:
    print('Chua tim thay output RFM. Hay kiem tra log cell chay pipeline.')

## Xem ket qua MBA (neu co)

In [ ]:
mba_rules_path = Path('data/3_curated/results/mba/association_rules')

if mba_rules_path.exists():
    mba_rules = pd.read_parquet(mba_rules_path)
    if mba_rules.empty:
        print('MBA output co ton tai nhung rong. Thu giam min-support roi chay lai.')
    else:
        display(mba_rules.sort_values('confidence', ascending=False).head(20))
else:
    print('Chua tim thay output MBA. Hay kiem tra log cell chay pipeline.')

## Ve bieu do nhanh

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

if 'rfm_segments' in globals() and not rfm_segments.empty:
    counts = (
        rfm_segments['segment_label']
        .value_counts()
        .rename_axis('segment_label')
        .reset_index(name='customers')
    )

    plt.figure(figsize=(8, 5))
    sns.barplot(data=counts, x='segment_label', y='customers', hue='segment_label', palette='Set2', legend=False)
    plt.title('So luong khach hang theo phan khuc')
    plt.xlabel('Phan khuc')
    plt.ylabel('So khach hang')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print('Chua co du lieu RFM de ve bieu do.')